# Recency window-length sweep on Colab GPU

Trains `gru_modelr` on windows of increasing length (all ending at the recent edge 1597) with a **fair, converged** epoch budget, on GPU with `batch_size=16` so the run is fast. Answers: how much recent data is optimal, and does older data help or hurt?

Rough time on a T4 at `batch_size=16`, `epochs=30`: the whole sweep (windows 50/100/200/400/898) ≈ **2 hours** (vs ~27 h on our laptop CPU at bs=1).

**Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi -L
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Data + project
Upload `kaggle.json` and the project zip (or `git clone` your repo) via the file browser first.

In [ ]:
# Kaggle data (~2 min)
!pip install -q kaggle
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c jane-street-real-time-market-data-forecasting -p /content/data
!cd /content/data && unzip -q -o '*.zip'
# Project (option A: unzip an uploaded project.zip; option B: git clone your repo)
!unzip -q -o project.zip -d /content/project 2>/dev/null || echo 'no project.zip — git clone your repo into /content/project'
%cd /content/project

In [ ]:
%pip install --quiet 'einops>=0.8' 'numpy>=2.0' 'polars>=1.20' 'pyarrow>=15' \
  'scikit-learn>=1.5' 'torch>=2.4' 'tqdm>=4.66' 'xgboost>=2.1' 'pyyaml>=6' 'iisignature>=0.24'
%pip install --quiet -e /content/project

## 2. Point config at the Colab data + precompute the pool (with lagged responders)
Rebuilds the Float16 memmap on Colab's local disk (~10 min). One memmap serves the whole sweep.

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# Patch the config's data_root to the Colab path (the default is the laptop path).
import janestreet.config as C
from pathlib import Path
C.Cfg.data_root = Path('/content/data')

!KMP_DUPLICATE_LIB_OK=TRUE python scripts/precompute_dataset.py \
  --min-date 700 --max-date 1698 --train-until 1598 \
  --lagged-responders 0,1,2,3,4,5,6,7,8 \
  --out /content/pool700_lags

## 3. Fair recency sweep on GPU
`gru_modelr`, `--device cuda --batch-size 16 --epochs 30 --patience 6` so every window fully converges. `batch_size=16` is what makes the GPU worthwhile (at bs=1 the RNN barely uses it).

In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python scripts/sweep_window_length.py \
  --data /content/pool700_lags --model gru_modelr \
  --lengths 50,100,200,400,898 --hi 1597 --pool-lo 700 \
  --valid-lo 1599 --valid-hi 1698 \
  --device cuda --batch-size 16 --epochs 30 --patience 6 \
  --out /content/window_sweep

## 4. Optional: soft recency (time-decay weighting) on the full pool
Instead of a hard window, down-weight old dates. Compare against the hard-cutoff curve above.

In [ ]:
# for HL in 60 120 240:
#   !KMP_DUPLICATE_LIB_OK=TRUE python scripts/train_from_memmap.py \
#     --data /content/pool700_lags --model gru_modelr --resample-mode window \
#     --train-lo 700 --train-hi 1597 --valid-lo 1599 --valid-hi 1698 \
#     --device cuda --batch-size 16 --epochs 30 --patience 6 \
#     --decay-halflife $HL --tag full_hl$HL --out /content/decay_sweep

## 5. Results
The sweep prints an online-R² recency curve and writes `sweep_summary.json`. Download it (and the per-window `.npz` in `/content/window_sweep/preds/`) to blend the winning window back into the laptop ensemble.

In [ ]:
import json
s = json.load(open('/content/window_sweep/sweep_summary.json'))
print('window   static      online')
for r in s['rows']:
    print(f"{r['length']:>5}d  {r['r2_static']:+.5f}  {r['r2_online']:+.5f}")
# download: left file browser → /content/window_sweep/ → right-click sweep_summary.json + preds/*.npz